In [2]:
library(MLmetrics)
library(randomForest)
set.seed(2) 

Warning message:
"le package 'MLmetrics' a été compilé avec la version R 4.2.3"

Attachement du package : 'MLmetrics'


L'objet suivant est masqué depuis 'package:base':

    Recall


Warning message:
"le package 'randomForest' a été compilé avec la version R 4.2.3"
randomForest 4.7-1.1

Type rfNews() to see new features/changes/bug fixes.



In [3]:
ConfusionMatrix <- function(y_pred, y_true) {
  Confusion_Mat <- table(y_true, y_pred)
  return(Confusion_Mat)
}
 
ConfusionDF <- function(y_pred, y_true) {
  Confusion_DF <- transform(as.data.frame(ConfusionMatrix(y_pred, y_true)),
                            y_true = as.character(y_true),
                            y_pred = as.character(y_pred),
                            Freq = as.integer(Freq))
  return(Confusion_DF)
}
 
Precision_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FP <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # it may happen that a label is never predicted (missing from y_pred) but exists in y_true
    # in this case ConfusionDF will not have these lines and thus the simplified code crashes
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]))
   
    # workaround:
    # i don't want to change ConfusionDF since i don't know if the current behaviour is a feature or a bug.
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
   
    tmp <- Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]
    FP[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Precision_micro <- sum(TP) / (sum(TP) + sum(FP))
  return(Precision_micro)
}
 
Recall_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FN <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # short version, comment out due to bug or feature of Confusion_DF
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]))
   
    # workaround:
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
 
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]
    FN[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Recall_micro <- sum(TP) / (sum(TP) + sum(FN))
  return(Recall_micro)
}
 
F1_Score_micro <- function(y_true, y_pred, labels = NULL) {
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred)) # possible problems if labels are missing from y_*
  Precision <- Precision_micro(y_true, y_pred, labels)
  Recall <- Recall_micro(y_true, y_pred, labels)
  F1_Score_micro <- 2 * (Precision * Recall) / (Precision + Recall)
  return(F1_Score_micro)
}

In [ ]:
data<-read.csv("train_values.csv",stringsAsFactors = T)
data_labels<-read.csv("train_labels.csv",stringsAsFactors = T)
datam<-merge(data,data_labels,by=c('building_id','building_id'))

In [4]:
datam<-read.csv("data_target_encoding.csv",stringsAsFactors = T)

In [5]:
ncol(datam)

[1] 40

We do not need to one hot encode categorical variables for randomForest. We will now scan for the optimal parameters n_trees and nb of features selected at each split. The function tuneRF could allow us to scan of nb of features selected at each split, but using was not optimal: we feel like it stopped too early in the search. Usual values of the parameter are sqrt(p), log2(p), ln(p) with p the amount of features.

**Should I try 1000 ?**

In [7]:
n_trees <- c(10,20,100,200,500)
nfeat <- ncol(datam)-2 #I remove 2 to remove building_id and damage_grade
m_tries <- c(floor(0.5*sqrt(nfeat)),floor(sqrt(nfeat)),floor(2*sqrt(nfeat)),nfeat)
accuracy_vec <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(accuracy_vec)<-c('n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- ncol(datam)-1 #I will remove id_variable too so I need to remove 1 col
id_variable <- match('building_id', colnames(datam))
pb <- txtProgressBar(min = 0, max = length(n_trees), style = 3)

for (i in n_trees){ 
    pb2 <- txtProgressBar(min = 0, max = length(m_tries), style = 3)
    for (j in m_tries){
        #3.1 Take the first half of the dataset as a training data set
        print(paste("Number of trees used:",i,'mTry:',j))
        train_data <- datam[datam_idx[1:split],-c(id_variable)]

        #3.2 Take the second half of the dataset as a hold out or test data set
        test_data <- datam[datam_idx[(split+1):nrows],-c(id_variable)]
        
        model <- randomForest(x=train_data[,-c(target_variable)],
                              y=as.factor(train_data[,c(target_variable)]),
                              ntree=i,mtry=j,doBest=TRUE,keep.forest=TRUE,importance=TRUE,do.trace=TRUE)
        #model<-tuneRF(x=train_data[,-c(target_variable),drop=F],
        #    y=as.factor(train_data[,c(target_variable)]),ntreeTry=i,mtryStart=12,plot=TRUE,trace=TRUE,doBest=TRUE,importance=TRUE,do.trace=TRUE)
        #model
        yhat<-predict(model,test_data[,-c(target_variable),drop=F])                      
        accuracy_vec[nrow(accuracy_vec)+1,]<-c(i,j,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
        print(paste("Best F1 Score - ",i, 'trees','- mtry',j,':',accuracy_vec[nrow(accuracy_vec),3],n=1))
        setTxtProgressBar(pb2, j)
    }
    setTxtProgressBar(pb, i)
}
print(accuracy_vec)
#write.csv(accuracy_vec,'randomForest_tuning.csv')

  |                                                                      |   0%[1] "Number of trees used: 10 mTry: 3"
ntree      OOB      1      2      3
    1:  39.79% 62.21% 18.84% 68.92%
    2:  39.10% 64.25% 15.85% 71.27%
    3:  38.03% 62.60% 16.87% 66.88%
    4:  37.68% 62.37% 16.89% 65.85%
    5:  37.49% 62.46% 16.65% 65.58%
    6:  37.08% 61.43% 16.71% 64.54%
    7:  36.65% 61.23% 16.16% 64.29%
    8:  36.19% 60.92% 15.65% 63.90%
    9:  36.06% 61.02% 15.28% 64.08%
   10:  35.58% 60.82% 14.96% 63.28%
[1] "Best F1 Score -  10 trees - mtry 3 : 10 1"               
[2] "Best F1 Score -  10 trees - mtry 3 : 3 1"                
[3] "Best F1 Score -  10 trees - mtry 3 : 0.664396308589628 1"
  |====================================================                  |  75%[1] "Number of trees used: 10 mTry: 6"
ntree      OOB      1      2      3
    1:  37.04% 55.99% 29.21% 44.72%
    2:  36.99% 55.56% 28.35% 46.20%
    3:  36.59% 55.19% 27.61% 46.49%
    4:  36.18% 55.26% 27.10% 46.09%

ERROR: Error in xy.coords(x, y, xlabel, ylabel, log): les longueurs de 'x' et 'y' diffèrent


In [ ]:
n_trees <- 1000
nfeat <- ncol(datam)-2 #I remove 2 to remove building_id and damage_grade
m_tries <- c(floor(0.5*sqrt(nfeat)),floor(sqrt(nfeat)),floor(1.5*sqrt(nfeat)),floor(2*sqrt(nfeat)),nfeat,14,16)
accuracy_vec <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(accuracy_vec)<-c('n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- ncol(datam)-1 #I will remove id_variable too so I need to remove 1 col
id_variable <- match('building_id', colnames(datam))
pb <- txtProgressBar(min = 0, max = length(n_trees), style = 3)

for (i in n_trees){ 
    pb2 <- txtProgressBar(min = 0, max = length(m_tries), style = 3)
    for (j in m_tries){
        #3.1 Take the first half of the dataset as a training data set
        print(paste("Number of trees used:",i,'mTry:',j))
        train_data <- datam[datam_idx[1:split],-c(id_variable)]

        #3.2 Take the second half of the dataset as a hold out or test data set
        test_data <- datam[datam_idx[(split+1):nrows],-c(id_variable)]
        
        model <- randomForest(x=train_data[,-c(target_variable)],
                              y=as.factor(train_data[,c(target_variable)]),
                              ntree=i,mtry=j,doBest=TRUE,keep.forest=TRUE,importance=TRUE,do.trace=TRUE)
        #model<-tuneRF(x=train_data[,-c(target_variable),drop=F],
        #    y=as.factor(train_data[,c(target_variable)]),ntreeTry=i,mtryStart=12,plot=TRUE,trace=TRUE,doBest=TRUE,importance=TRUE,do.trace=TRUE)
        #model
        yhat<-predict(model,test_data[,-c(target_variable),drop=F])                      
        accuracy_vec[nrow(accuracy_vec)+1,]<-c(i,j,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
        print(paste("Best F1 Score - ",i, 'trees','- mtry',j,':',accuracy_vec[nrow(accuracy_vec),3],n=1))
        setTxtProgressBar(pb2, j)
    }
    setTxtProgressBar(pb, i)
}
print(accuracy_vec)
#write.csv(accuracy_vec,'randomForest_tuning.csv')

  |                                                                      |   0%[1] "Number of trees used: 1000 mTry: 3"
ntree      OOB      1      2      3
    1:  39.79% 62.21% 18.84% 68.92%
    2:  39.10% 64.25% 15.85% 71.27%
    3:  38.03% 62.60% 16.87% 66.88%
    4:  37.68% 62.37% 16.89% 65.85%
    5:  37.49% 62.46% 16.65% 65.58%
    6:  37.08% 61.43% 16.71% 64.54%
    7:  36.65% 61.23% 16.16% 64.29%
    8:  36.19% 60.92% 15.65% 63.90%
    9:  36.06% 61.02% 15.28% 64.08%
   10:  35.58% 60.82% 14.96% 63.28%
   11:  35.31% 60.63% 14.27% 63.73%
   12:  35.37% 61.16% 13.84% 64.47%
   13:  34.95% 60.63% 13.50% 63.97%
   14:  35.04% 61.03% 12.93% 65.07%
   15:  34.82% 61.03% 13.02% 64.29%
   16:  34.55% 61.06% 12.63% 64.13%
   17:  34.58% 61.44% 12.54% 64.27%
   18:  34.39% 61.55% 12.40% 63.90%
   19:  34.02% 61.65% 12.21% 63.10%
   20:  33.95% 61.75% 11.87% 63.45%
   21:  33.99% 61.71% 11.25% 64.60%
   22:  34.05% 61.86% 11.04% 65.12%
   23:  33.91% 62.00% 10.98% 64.74%
   24:  33.90% 6

: 

: 

In [5]:
n_trees <- c(500,750)
nfeat <- ncol(datam)-2 #I remove 2 to remove building_id and damage_grade
m_tries <- c(floor(sqrt(nfeat)),floor(1.5*sqrt(nfeat)),floor(2*sqrt(nfeat)))
accuracy_vec <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(accuracy_vec)<-c('n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- ncol(datam)-1 #I will remove id_variable too so I need to remove 1 col
id_variable <- match('building_id', colnames(datam))
pb <- txtProgressBar(min = 0, max = length(n_trees), style = 3)

for (i in n_trees){ 
    pb2 <- txtProgressBar(min = 0, max = length(m_tries), style = 3)
    for (j in m_tries){
        #3.1 Take the first half of the dataset as a training data set
        print(paste("Number of trees used:",i,'mTry:',j))
        train_data <- datam[datam_idx[1:split],-c(id_variable)]

        #3.2 Take the second half of the dataset as a hold out or test data set
        test_data <- datam[datam_idx[(split+1):nrows],-c(id_variable)]
        
        model <- randomForest(x=train_data[,-c(target_variable)],
                              y=as.factor(train_data[,c(target_variable)]),
                              ntree=i,mtry=j,doBest=TRUE,keep.forest=TRUE,importance=TRUE,do.trace=TRUE)
        #model<-tuneRF(x=train_data[,-c(target_variable),drop=F],
        #    y=as.factor(train_data[,c(target_variable)]),ntreeTry=i,mtryStart=12,plot=TRUE,trace=TRUE,doBest=TRUE,importance=TRUE,do.trace=TRUE)
        #model
        yhat<-predict(model,test_data[,-c(target_variable),drop=F])                      
        accuracy_vec[nrow(accuracy_vec)+1,]<-c(i,j,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
        print(paste("Best F1 Score - ",i, 'trees','- mtry',j,':',accuracy_vec[nrow(accuracy_vec),3],n=1))
        setTxtProgressBar(pb2, j)
    }
    setTxtProgressBar(pb, i)
}
print(accuracy_vec)
#write.csv(accuracy_vec,'randomForest_tuning.csv')

  |                                                                      |   0%

In [ ]:
write.csv(accuracy_vec,'randomForest_tuning_temp.csv')

from here: target encoded

In [ ]:
n_trees <- c(10,20,100,200,500)
nfeat <- ncol(datam)-2 #I remove 2 to remove building_id and damage_grade
m_tries <- c(floor(0.5*sqrt(nfeat)),floor(sqrt(nfeat)),floor(2*sqrt(nfeat)),nfeat)
accuracy_vec <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(accuracy_vec)<-c('n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- ncol(datam)-1 #I will remove id_variable too so I need to remove 1 col
id_variable <- match('building_id', colnames(datam))
pb <- txtProgressBar(min = 0, max = length(n_trees), style = 3)

for (i in n_trees){ 
    pb2 <- txtProgressBar(min = 0, max = length(m_tries), style = 3)
    for (j in m_tries){
        #3.1 Take the first half of the dataset as a training data set
        print(paste("Number of trees used:",i,'mTry:',j))
        train_data <- datam[datam_idx[1:split],-c(id_variable)]

        #3.2 Take the second half of the dataset as a hold out or test data set
        test_data <- datam[datam_idx[(split+1):nrows],-c(id_variable)]
        
        model <- randomForest(x=train_data[,-c(target_variable)],
                              y=as.factor(train_data[,c(target_variable)]),
                              ntree=i,mtry=j,doBest=TRUE,keep.forest=TRUE,importance=TRUE,do.trace=TRUE)
        #model<-tuneRF(x=train_data[,-c(target_variable),drop=F],
        #    y=as.factor(train_data[,c(target_variable)]),ntreeTry=i,mtryStart=12,plot=TRUE,trace=TRUE,doBest=TRUE,importance=TRUE,do.trace=TRUE)
        #model
        yhat<-predict(model,test_data[,-c(target_variable),drop=F])                      
        accuracy_vec[nrow(accuracy_vec)+1,]<-c(i,j,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
        print(paste("Best F1 Score - ",i, 'trees','- mtry',j,':',accuracy_vec[nrow(accuracy_vec),3],n=1))
        setTxtProgressBar(pb2, j)
        rm('model')
    }
    setTxtProgressBar(pb, i)
}
print(accuracy_vec)
#write.csv(accuracy_vec,'randomForest_tuning.csv') 

  |                                                                      |   0%[1] "Number of trees used: 10 mTry: 3"
ntree      OOB      1      2      3
    1:  29.04% 49.40% 21.17% 36.49%
    2:  28.84% 48.57% 20.55% 37.30%
    3:  28.83% 47.25% 20.32% 38.04%
    4:  29.08% 48.20% 19.72% 39.55%
    5:  29.77% 48.03% 18.15% 44.29%
    6:  29.03% 47.47% 18.01% 42.47%
    7:  28.56% 48.57% 17.01% 42.48%
    8:  28.20% 48.66% 16.71% 41.92%
    9:  27.68% 48.13% 16.44% 40.95%
   10:  27.37% 48.67% 15.73% 41.10%
[1] "Best F1 Score -  10 trees - mtry 3 : 0.748837837316513 1"
  |====================================================                  |  75%[1] "Number of trees used: 10 mTry: 6"
ntree      OOB      1      2      3
    1:  31.18% 47.73% 25.78% 35.72%
    2:  31.32% 47.22% 25.38% 36.93%
    3:  31.01% 45.87% 24.86% 37.27%
    4:  30.67% 45.45% 24.26% 37.35%
    5:  30.35% 44.71% 23.75% 37.47%
    6:  29.89% 44.47% 23.12% 37.24%
    7:  29.60% 44.21% 22.76% 37.07%
    8:  29.15% 44

: 

: 